## **LoRA Fine-Tuning — Text-to-SQL (Improved)**

This notebook fine-tunes a small open LLM using pure LoRA (no QLoRA / no 4-bit quantization) so you can see the standard LoRA workflow clearly.

**Improvements over the original version:**
- Reproducible runs via a fixed seed
- LoRA also targets the MLP projections (`gate/up/down_proj`), not just attention — usually a quality win for a small extra param cost
- A real **exact-match accuracy** check on held-out data, not just training/eval loss (loss tells you it learned *a* pattern, not that the SQL is *correct*)
- `load_best_model_at_end` so you keep the best checkpoint instead of just the last one
- A single reusable `run_probes()` helper instead of duplicating the before/after probe-printing code
- Removed a no-op Llama-1/2 tensor-parallel config flag that does nothing for TinyLlama

**Steps:**
1. GPU check (T4 = 16 GB VRAM)
2. Load a base model in FP16
3. Inspect how the *base* model answers prompts (before fine-tuning)
4. Prepare a small instruction dataset (keeping a raw copy for evaluation)
5. Attach LoRA adapters and train
6. Inspect how the *fine-tuned* model answers the same prompts (after)
7. Measure exact-match accuracy on held-out examples
8. Save adapters

> **Base model:** `TinyLlama/TinyLlama-1.1B-Chat-v1.0` — ~1.1B params, ~2.2 GB in FP16.


## **1. Environment Setup**

In [ ]:
# install the required libraries
!pip install -q \
  "transformers==5.0.0" \
  "peft==0.18.1" \
  "accelerate==1.13.0" \
  "datasets==4.8.4" \
  "trl==1.1.0" \
  "sentencepiece==0.2.1" \
  "protobuf==5.29.6"


In [ ]:
import sys
import random
import numpy as np
import torch  # pre-installed in colab env

# --- Reproducibility -------------------------------------------------------
SEED = 42

def set_all_seeds(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)

print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__)

# check GPU availability
print("CUDA avail :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name   :", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory : {props.total_memory / 1024**3:.2f} GB")

# Expected on Colab free tier: Tesla T4, ~14.5 GB


## **2. Load the base model in FP16**

We load the model in half precision (`torch.float16`). On T4 this is the sweet spot — faster than FP32, and the model comfortably fits in memory alongside a small LoRA adapter.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False   # required for gradient checkpointing later
# Note: `pretraining_tp` (tensor-parallel degree) is a Llama-1/2-specific flag
# and is a no-op for TinyLlama, so it's intentionally omitted here.

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params/1e6:.1f} M")
print(f"Memory footprint: {model.get_memory_footprint()/1024**3:.2f} GB")


## **3. Define a prompt format and test the BASE model**

We use TinyLlama's chat template. Task = text-to-SQL. Given a table schema and a question, produce a SQL query.

In [ ]:
def build_prompt(schema: str, question: str) -> str:
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    user = f"Schema:\n{schema}\n\nQuestion: {question}"
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [ ]:
# testing build_prompt function
test_prompt_1 = build_prompt(
    schema="CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
    question="List the names of employees in the Engineering department earning more than 100000."
)

print(test_prompt_1)


In [ ]:
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 120) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Slice only newly generated tokens
    input_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][input_length:]

    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


In [ ]:
# Three probe prompts we will reuse AFTER fine-tuning for direct comparison.
PROBES = [
    {
        "schema":   "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
        "question": "List the names of employees in the Engineering department earning more than 100000.",
    },
    {
        "schema":   "CREATE TABLE orders (order_id INT, customer_id INT, amount FLOAT, order_date DATE);",
        "question": "What is the total order amount per customer in 2024?",
    },
    {
        "schema":   "CREATE TABLE movies (title TEXT, year INT, rating FLOAT, genre TEXT);",
        "question": "Show the top 5 highest rated horror movies released after 2015.",
    },
]

def run_probes(label: str, previous_outputs=None):
    # Run PROBES through the current global `model` and print results.
    # If `previous_outputs` is given, prints a BEFORE/AFTER comparison instead.
    # Returns the list of generated outputs (so you can pass them in next time).
    print("=" * 70)
    print(label)
    print("=" * 70)
    outputs = []
    for i, p in enumerate(PROBES, 1):
        prompt = build_prompt(p["schema"], p["question"])
        ans = generate(prompt)
        outputs.append(ans)
        print(f"\n--- Probe {i} ---")
        print("Q:     ", p["question"])
        if previous_outputs is not None:
            print("BEFORE:", previous_outputs[i - 1])
            print("-" * 70)
            print("AFTER :", ans)
        else:
            print("A:", ans)
        print("=" * 70)
    return outputs

base_outputs = run_probes("BASE MODEL (before fine-tuning)")


Expect the base model to *talk about* the query, add commentary, re-explain the schema, or produce malformed SQL. That is the "before" state.

## **4. Load and prepare the dataset**

We use `b-mc2/sql-create-context` — a compact text-to-SQL dataset with `(question, context, answer)` triples. We'll take a small slice so training finishes in a few minutes on T4.

We keep a **raw, unformatted copy of the eval split** (`eval_raw`) around — we need the plain `question`/`context`/`answer` fields later for exact-match scoring, and that information is gone once we flatten everything into a single chat-template `text` string.

In [ ]:
from datasets import load_dataset

raw = load_dataset("b-mc2/sql-create-context", split="train")
print("Full dataset size:", len(raw))
print("Example row     :", raw[0])

# Keep it small for a fast, visible demo on T4
raw = raw.shuffle(seed=SEED).select(range(3000))
split = raw.train_test_split(test_size=0.05, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

# Keep an untouched copy of eval before we reformat it into chat-template text —
# we need question/context/answer separately for exact-match evaluation later.
eval_raw = eval_ds

print("Train:", len(train_ds), " Eval:", len(eval_ds))


In [ ]:
def format_example(row):
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    user      = f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"
    assistant = row["answer"]
    messages = [
        {"role": "system",    "content": system},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": assistant},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

train_ds_fmt = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds_fmt  = eval_ds.map (format_example, remove_columns=eval_ds.column_names)

print("\n--- Formatted training example ---\n")
print(train_ds_fmt[0]["text"][:800])


## **5. Attach LoRA Adapters**

LoRA inserts low-rank trainable matrices into specific linear layers while freezing the base weights.

We target the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`) **and** the MLP projections (`gate_proj`, `up_proj`, `down_proj`). Attention-only is the common minimal setup, but including the MLP layers usually improves downstream quality noticeably for a modest increase in trainable parameters — worth the trade-off here since our dataset and training budget are both small.

In [ ]:
from peft import LoraConfig, get_peft_model

# Enable gradient checkpointing to save VRAM during backward pass
model.gradient_checkpointing_enable()
model.enable_input_require_grads()     # needed because base params are frozen

lora_config = LoraConfig(
    r=16,                                # rank
    lora_alpha=32,                       # scaling factor (alpha / r = 2.0)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",      # MLP
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## **6. Train**

We use `SFTTrainer` from TRL.

Settings chosen for T4 (16 GB):
- `per_device_train_batch_size=2`, `gradient_accumulation_steps=8` → effective batch 16
- `max_seq_length=512`
- `fp16=True` (T4 supports FP16, not BF16)
- 1 epoch over 3000 examples ≈ ~5–10 minutes
- `load_best_model_at_end=True` so the checkpoint you end up with is the one with the lowest eval loss, not just whatever the last step happened to produce — this matters more once you extend past 1 epoch


In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./tinyllama-sql-lora"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=SEED,
)

# Pre-tokenize so we don't depend on SFTTrainer's text-field handling
def tokenize(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )
    return out

train_tok = train_ds_fmt.map(tokenize, batched=True, remove_columns=["text"])
eval_tok  = eval_ds_fmt.map (tokenize, batched=True, remove_columns=["text"])

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer,
)

trainer.train()


In [ ]:
# Peak VRAM used during training — should stay well under T4's 15.8 GB
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")


### **SFTConfig Parameters Explained**

- `output_dir=OUTPUT_DIR` — Folder where the fine-tuned model, checkpoints, and logs will be saved.
- `num_train_epochs=1` — Number of times the model sees the full training dataset.
- `per_device_train_batch_size=2` — Number of training samples processed at once on each device.
- `per_device_eval_batch_size=2` — Number of evaluation samples processed at once on each device.
- `gradient_accumulation_steps=8` — Accumulates gradients for 8 steps before updating weights, which gives an effective larger batch size.
- `gradient_checkpointing=True` — Saves GPU memory by recomputing some activations during backpropagation.
- `learning_rate=2e-4` — Controls how big each weight update is during training.
- `lr_scheduler_type="cosine"` — Gradually changes the learning rate using a cosine decay schedule.
- `warmup_steps=10` — Slowly increases the learning rate for the first 10 steps to make training stable.
- `optim="adamw_torch"` — Optimizer used to update model weights, here it is AdamW from PyTorch.
- `fp16=True` — Uses 16-bit floating point precision to reduce memory usage and speed up training.
- `bf16=False` — Disables bfloat16 precision.
- `logging_steps=10` — Logs training details every 10 steps.
- `eval_strategy="steps"` — Runs evaluation during training based on step intervals.
- `eval_steps=50` — Evaluates the model every 50 steps.
- `save_strategy="steps"` / `save_steps=50` — Saves a checkpoint every 50 steps instead of only once per epoch, so `load_best_model_at_end` has multiple checkpoints to choose from.
- `save_total_limit=2` — Keeps disk usage bounded by only retaining the 2 most recent/best checkpoints.
- `load_best_model_at_end=True` / `metric_for_best_model="eval_loss"` — After training, reloads the checkpoint with the lowest eval loss rather than just keeping the final step's weights.
- `report_to="none"` — Disables reporting to tools like WandB or TensorBoard.
- `seed=SEED` — Makes the trainer's own shuffling/sampling reproducible.


## **7. Compare: AFTER Fine-Tuning**

In [ ]:
# Re-enable cache for fast inference
model.config.use_cache = True
model.eval()

_ = run_probes("FINE-TUNED MODEL (after LoRA)", previous_outputs=base_outputs)


You should now see:
- **Before:** verbose, chatty, often wrong syntax, re-explains the schema.
- **After:** terse, well-formed SQL — the model learned the response *style* of the dataset.

That style shift is exactly what LoRA buys you for a few minutes of T4 compute. But style ≠ correctness — that's what the next section actually measures.

## **8. Exact-Match Evaluation (NEW)**

Training/eval **loss** only tells you the model is predicting tokens closer to the reference — it does not tell you the generated SQL is actually correct. This section runs the fine-tuned model on every held-out example and checks whether the generated SQL exactly matches the reference answer (case-insensitive, whitespace-normalized).

Exact-match is a strict, cheap proxy — semantically equivalent SQL with different formatting/aliasing will count as wrong. For a more forgiving signal you could add execution-based evaluation (run both queries against a real/mock DB and diff the result rows), but exact-match is a good first rigor check and needs no extra infrastructure.

In [ ]:
import re

def normalize_sql(sql: str) -> str:
    sql = sql.strip().rstrip(";").strip()
    sql = re.sub(r"\s+", " ", sql)
    return sql.lower()

def evaluate_exact_match(dataset, max_examples: int = None, verbose_mismatches: int = 3):
    n = len(dataset) if max_examples is None else min(max_examples, len(dataset))
    correct = 0
    mismatches_shown = 0
    for i in range(n):
        ex = dataset[i]
        prompt = build_prompt(ex["context"], ex["question"])
        pred = generate(prompt)
        is_match = normalize_sql(pred) == normalize_sql(ex["answer"])
        correct += int(is_match)
        if not is_match and mismatches_shown < verbose_mismatches:
            print("--- Mismatch example ---")
            print("Q:        ", ex["question"])
            print("Expected: ", ex["answer"])
            print("Predicted:", pred)
            print()
            mismatches_shown += 1
    acc = correct / n
    print(f"\nExact-match accuracy on {n} held-out examples: {acc:.1%} ({correct}/{n})")
    return acc

exact_match_accuracy = evaluate_exact_match(eval_raw)


If accuracy looks low, remember the caveat from the intro: with a 1.1B base model, 3,000 training examples, and 1 epoch, the model reliably learns *format* but not full task mastery — that's expected here, not a bug in the notebook. Scaling up data/epochs/model size (see the Recap section) is what closes that gap.

## **9. Save the Adapters**

LoRA adapters are tiny (a few MB). You save *only* the adapter, not the whole base model.

In [ ]:
ADAPTER_DIR = "./tinyllama-sql-lora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
total = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"Adapter size on disk: {total/1024**2:.2f} MB")


## **Loading the adapter later (for reference)**

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, "./tinyllama-sql-lora-adapter")
tokenizer = AutoTokenizer.from_pretrained("./tinyllama-sql-lora-adapter")
```


## Recap

| Step | What happened |
|---|---|
| Base model loaded in FP16 | ~2.2 GB VRAM |
| LoRA adapters attached (r=16, attention + MLP projections) | ~1% trainable params |
| Trained 1 epoch on 3000 SQL examples | ~5–10 min on T4, peak VRAM well under 16 GB |
| Best checkpoint reloaded via `load_best_model_at_end` | avoids keeping an overfit last-step checkpoint |
| Exact-match accuracy measured on held-out examples | rigor check beyond loss curves |
| Adapter saved | A few MB on disk |
| Visible outcome | Chatty base → clean SQL after fine-tuning |

**To extend this:** swap the dataset (e.g. `databricks/databricks-dolly-15k`, a style-transfer set, or your own JSONL), keep the same LoRA config, and you've got the same workflow for any instruction-following task that fits in T4.


---

**Note on accuracy**

This notebook is a demonstration of the LoRA workflow, not a production SQL model. With a small 1.1B base, 3,000 examples, and 1 epoch, the fine-tuned model reliably learns the response style (clean SQL instead of chatty explanations) but is not always semantically correct — the exact-match score in Section 8 makes that gap explicit instead of leaving it implied by loss curves alone.

To improve accuracy, the same workflow scales up: use a larger base model (e.g. Qwen2.5-3B, or a 7B with QLoRA), more training data (the full dataset has ~78k examples), 2–3 epochs instead of 1, and consider execution-based (not just exact-match) evaluation once you're optimizing for real accuracy rather than a demo.

---
